# Solving — Why Backtracking with MRV

This notebook walks through **why** the Sudoku solver is MRV-ordered backtracking (and not simulated annealing, a genetic algorithm, or full constraint propagation), **how** the algorithm compares to the classical variant in Kamal et al. [\[1\]](#references), and **why** 3 of the 38 ground-truth puzzles legitimately fail the benchmark.

**Prerequisites.** Python + the solver module — no ML dependencies needed.

**Related reading**

- `../app/core/solver.py:23` — production `backtracking` implementation (matches the MRV pseudocode below line for line)
- `../evaluation/benchmark_solver.py` — latency benchmark against the 38-puzzle GT set
- `../evaluation/solver_benchmark_results.json` — committed benchmark output
- **[1]** Kamal, Chawla & Goel, *Detection of Sudoku Puzzle using Image Processing and Solving by Backtracking, Simulated Annealing and Genetic Algorithms* (ICIIP 2015) — the comparative study whose Algorithm 1 is the skeleton this solver extends
- **[2]** Bhattarai, Uprety, Pathak, Shrestha, Narkarmi & Sigdel, *A Study Of Sudoku Solving Algorithms: Backtracking and Heuristic* (arXiv:2507.09708, 2025) — a more recent comparison showing constraint-propagation heuristics (AC-3, naked/hidden singles) are 1.27× to 2.91× faster than pure recursive backtracking on expert-level puzzles


In [ ]:
import sys
import json
import time
import statistics
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.core.solver import backtracking, solve


## 1. Sudoku as a constraint satisfaction problem (CSP)

A 9×9 Sudoku is a CSP with 81 variables (cells), each with domain `{1, …, 9}`. The constraints are: each row contains each digit once, each column contains each digit once, each 3×3 box contains each digit once. A "solution" is an assignment of values to all variables that satisfies every constraint. A well-formed puzzle has exactly one solution.

The search space for a naive generator is enormous: 9^81 ≈ 2 × 10^77 assignments. But the constraint structure is dense — every cell is involved in 3 constraints (row + column + box) — so **pruning dominates generation**. Any practical solver must propagate constraints to shrink the search tree, and the quality of the pruning strategy is what separates fast solvers from slow ones.

Kamal et al. [\[1\]](#references) compare three classical approaches on 9×9 Sudoku: recursive backtracking (Algorithm 1 in their paper), simulated annealing (Algorithm 2), and a genetic algorithm (Algorithm 3). Their conclusion is unambiguous: backtracking wins across every difficulty level they test. Simulated annealing's performance degrades sharply beyond the 27-clue threshold (essentially when the number of empty cells exceeds 54); the genetic algorithm fails to converge within a reasonable budget for any hard puzzle. **That's the reason this repo ships only backtracking** — the elaborate stochastic alternatives don't pay for themselves at this problem size. See section 7 below for the removal history.


## 2. Classical recursive backtracking (Kamal et al., Algorithm 1)

The textbook procedure, paraphrased from the paper:

```
Algorithm 1  Backtracking (Kamal et al. 2015)

 1:  procedure BACKTRACKING(G, N)
 2:      Input:  Sudoku puzzle G, order N (3 for 9×9)
 3:      function SOLVE
 4:          if Find an empty cell then
 5:              for Number = 1 to N² do
 6:                  if Safe to fill the Number then
 7:                      Fill the empty cell with Number
 8:                      if SOLVE then
 9:                          Output: Sudoku solved
10:                      end if
11:                      Empty the cell
12:                  end if
13:              end for
14:              Return False
15:          else
16:              Output: Sudoku solved
17:          end if
18:      end function
19:  end procedure
```

This is correct but wasteful. The empty cell at line 4 is the *first* empty cell found (row-major scan); the branching factor is up to 9 per empty cell regardless of how constrained the cell actually is. On a 30-empty-cell puzzle the worst-case search tree has up to 9^30 ≈ 4.2 × 10^28 leaves.

A reference Python implementation would look like this (not the production code — the production solver adds MRV):


In [ ]:
def backtracking_naive(puzzle):
    """Plain recursive backtracking, row-major order — the Kamal et al. Algorithm 1 baseline."""
    grid = [row[:] for row in puzzle]
    nodes = [0]

    def is_safe(r, c, val):
        for k in range(9):
            if grid[r][k] == val or grid[k][c] == val:
                return False
        br, bc = 3 * (r // 3), 3 * (c // 3)
        for i in range(br, br + 3):
            for j in range(bc, bc + 3):
                if grid[i][j] == val:
                    return False
        return True

    def solve_inner():
        for r in range(9):
            for c in range(9):
                if grid[r][c] == 0:
                    for val in range(1, 10):
                        nodes[0] += 1
                        if is_safe(r, c, val):
                            grid[r][c] = val
                            if solve_inner():
                                return True
                            grid[r][c] = 0
                    return False
        return True

    success = solve_inner()
    return grid, nodes[0], success


## 3. The MRV extension

MRV (Minimum Remaining Value) is a standard heuristic in CSP literature: at each recursive step, expand the variable with the *smallest* current domain first. For Sudoku that means: scan the empty cells, compute each cell's candidate set `{1..9} − row − col − box`, and recurse on the cell with the fewest candidates.

**Why it helps.** Two effects stack:

1. **Forced moves collapse.** Any cell with exactly one candidate is a forced move — the solver fills it immediately and moves on, with zero branching. Naive row-major backtracking would still try all 9 values at that cell in order.
2. **Branching factor shrinks everywhere else.** The smallest-domain cell has branching factor ≤ its domain size, not 9. On a hard puzzle this is the difference between a 9^n tree and a 2^n or 3^n tree.

A bonus: the solver detects dead ends earlier. If the smallest-domain cell has an empty candidate set, the current partial assignment is already infeasible and we return False immediately — before spending recursion on any other cell.


## 4. The production MRV backtracking pseudocode

This is the exact pseudocode for what's in `app/core/solver.py:23`, in the same Kamal-et-al. academic format:

```
Algorithm 1  MRV-Ordered Backtracking for Sudoku

 1:  procedure Solve(G)
 2:      Input:   9 × 9 grid G with 0 denoting empty cells
 3:      Output:  True if a solution is written into G in place; False otherwise
 4:
 5:      best       ← None
 6:      best_cands ← None
 7:
 8:      for each cell (r, c) in G do
 9:          if G[r][c] = 0 then
10:              cands ← {1, …, 9} − used_in_row(r) − used_in_col(c) − used_in_box(r, c)
11:              if cands = ∅ then
12:                  return False                                 ▷ dead end: backtrack
13:              end if
14:              if best = None or |cands| < |best_cands| then
15:                  best       ← (r, c)
16:                  best_cands ← cands
17:                  if |cands| = 1 then
18:                      break                                    ▷ forced move: stop scanning
19:                  end if
20:              end if
21:          end if
22:      end for
23:
24:      if best = None then
25:          return True                                          ▷ grid is full: puzzle solved
26:      end if
27:
28:      (r, c) ← best
29:      for each val in best_cands do
30:          G[r][c] ← val
31:          if Solve(G) then
32:              return True
33:          end if
34:          G[r][c] ← 0                                          ▷ undo assignment
35:      end for
36:      return False
37:  end procedure
```

Compared to Kamal's Algorithm 1: we've added lines 5–6 (best-cell tracking), lines 10–20 (candidate computation and MRV selection with early termination on forced moves), and line 12 (early dead-end detection). Everything else is structurally the same.


## 5. Benchmark — latency on the 38-puzzle GT set

The committed benchmark file `evaluation/solver_benchmark_results.json` contains results from running `python -m evaluation.benchmark_solver`, which times the production `backtracking` function 10 times per puzzle and reports min / median / mean / p95 / max across all runs.


In [ ]:
bench_path = PROJECT_ROOT / "evaluation" / "solver_benchmark_results.json"
if not bench_path.exists():
    print("Benchmark results not found. Run: python -m evaluation.benchmark_solver")
else:
    data = json.loads(bench_path.read_text())
    bt = data["backtracking"]
    solvable = bt["solvable_runs"] or bt["all_runs"]
    print(f"Runs per puzzle : {bt['runs_per_puzzle']}")
    print(f"Total runs      : {bt['total_runs']}")
    print(f"Puzzles total   : {bt['puzzles_total']}")
    print(f"Puzzles solvable: {bt['puzzles_solvable']}")
    print(f"Failed (GT bad) : {bt['failed_paths']}")
    print()
    print("Latency on solvable subset:")
    for k in ["min_ms", "median_ms", "mean_ms", "p95_ms", "max_ms"]:
        print(f"  {k:<12} {solvable[k]:>8.4f}")


## 6. The 3 unsolvable puzzles — a data-quality signal, not a solver bug

The benchmark reports 35 of 38 puzzles as solvable. The 3 failures aren't solver regressions — they're *ground-truth data errors*. Specifically, the clue grids I hand-entered for those three images contain duplicate digits in a single row, column, or box, which makes them unsolvable as a CSP by construction.

This is a **feature**, not a bug, of the benchmark harness. The solver correctly rejects infeasible inputs; the harness bubbles the failure up as a "failed path" in the output JSON; I noticed the 3/38 failure rate and re-checked the GT file rather than blaming the solver. Fixing the annotations is on the roadmap (re-annotate via `annotate.py --redo`), but the latency number is stable across the fix: the full-38 median and the solvable-35 median both round to 0.42 ms because the 3 failures terminate quickly on the first contradiction.

**Takeaway:** a benchmark harness that catches errors in the evaluation data is doing its job on two axes — regression protection AND data-quality signalling. This is why every metric in the main README's Benchmarks table is tied to a `python -m evaluation.*` command rather than a static number.


In [ ]:
# Show the contradiction in one of the flagged puzzles (purely for illustration)
if bench_path.exists():
    data = json.loads(bench_path.read_text())
    failed = data["backtracking"]["failed_paths"]
    if failed:
        first_bad_path = failed[0]
        with open(PROJECT_ROOT / "evaluation" / "ground_truth.json") as f:
            gt = json.load(f)["images"]
        entry = next(e for e in gt if e["path"] == first_bad_path)
        grid = [[c if not isinstance(c, list) else 0 for c in row] for row in entry["grid"]]
        print(f"First flagged puzzle: {first_bad_path}")
        print()
        for row in grid:
            print(" ".join(str(c) if c else "." for c in row))
        # Find the duplicate
        for i, row in enumerate(grid):
            clues = [c for c in row if c != 0]
            if len(clues) != len(set(clues)):
                dup = [x for x in set(clues) if clues.count(x) > 1]
                print(f"\nRow {i} has duplicate digits: {dup}")
                break


## 7. Why we dropped simulated annealing

Earlier versions of `solver.py` shipped a second `simulated_annealing` function alongside `backtracking`. The SA implementation was textbook: per-box initialization (fill each 3×3 box with a permutation of 1–9, so box constraints are satisfied by construction), box-internal swap neighbours (swap two non-fixed cells within a single box so the box constraint is preserved), a row+column conflict energy function, geometric cooling `T = T₀ × 0.99999^n`, and periodic reheating to `0.5 × T₀` every 100,000 iterations when stuck.

The comparative study in Kamal et al. [\[1\]](#references) covers exactly this approach as their Algorithm 2, and the paper's Figure 6 plots backtracking vs SA vs GA vs puzzle complexity. The verdict: SA is competitive on easy puzzles (27+ clues) but "performance of simulated annealing degrades sharply when complexity exceeds 27" (their phrase) — it starts hitting timeouts on the harder GT images. Backtracking with MRV solves every solvable GT puzzle in under 5 ms.

The SA code was removed in the 2026-04-10 cleanup pass because:

1. It wasn't outperforming backtracking on any benchmark I could measure.
2. Keeping two solvers meant a `method` parameter in the API (`SolveRequest.method`) and a `simulated_annealing` entry in the `/api/health` solver list — extra surface area for no functional win.
3. Kamal et al.'s empirical result directly contradicts "keep SA for diversity" — they ran the comparison and backtracking won.

If you want to see the old SA implementation, it's in the git history — search for commits before `2026-04-10` in `app/core/solver.py`.

## 8. Future work — constraint propagation

The next obvious speedup beyond MRV is **constraint propagation**: iteratively apply inference rules like arc-consistency (AC-3), naked singles, hidden singles, naked pairs, and so on, to shrink every cell's candidate set *before* branching. The 2025 comparative study by Bhattarai et al. [\[2\]](#references) implements a heuristic-based Sudoku solver using constraint propagation and compares it against pure recursive backtracking on 500 puzzles across five difficulty levels (Beginner through Expert). They report speedups of **1.27× (Beginner) to 2.91× (Expert)** for constraint propagation over pure backtracking.

Their "Heuristic" solver is still built on recursive backtracking at the leaves — propagation just reduces the number of branches. So the path from the current MRV solver to a constraint-propagation solver is additive, not replacement: keep the MRV + backtracking skeleton, add an AC-3 step before each recursive call, add naked/hidden-single inference, measure. On expert-level puzzles (20 clues) the benefit is 2.91×; on the 38-image GT set (which is mostly middle-difficulty) the expected gain is closer to 1.5–2×, and the implementation cost is real. That's why it's a roadmap item and not a shipped feature.

## <a id="references"></a>References

**[1]** S. Kamal, S. S. Chawla, and N. Goel, "Detection of Sudoku Puzzle using Image Processing and Solving by Backtracking, Simulated Annealing and Genetic Algorithms: A Comparative Analysis," in *2015 Third International Conference on Image Information Processing (ICIIP)*, IEEE, Dec. 2015, pp. 179–184.

**[2]** A. Bhattarai, D. Uprety, P. Pathak, S. N. Shrestha, S. Narkarmi, and S. Sigdel, "A Study Of Sudoku Solving Algorithms: Backtracking and Heuristic," *arXiv preprint* arXiv:2507.09708, July 2025.
